## Learning notes — 01.Dynamic Views for Data Level Security

### Learning objectives
Follow **01.Dynamic Views for Data Level Security** from its source inputs through its operations to the archived results.
- Explain schema, security, ctas, write, profiling as used in this lesson.

### Source-specific learning focus
- Compare raw fields with typed records and parsing failures.
- Explain how identity and governance affect visible data and permitted actions.
- Distinguish query results from materialized table state.
- Trace how source records change persistent destination state.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `sales_dynamic`, `vw_sales_dynamic`.


# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

# Data-Level Security Using Dynamic Views

This notebook demonstrates how data-level security can be implemented using dynamic views.

The implementation covers two key capabilities:

- Row-Level Security: Controls which rows are visible to a user  
- Column-Level Masking: Controls how sensitive values are displayed  

A simple sales dataset is used to illustrate how different users receive different results when querying the same view.

#### Step 1: Set up the schema for the table

## Learning notes — before source operation

**schema**: Compare declared or inferred field types with raw input. Parsing, schema evolution and rescued/corrupt records can produce nulls or additional columns rather than clean typed rows.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Parsing a numeric field from an incompatible string can fail or yield null depending on parsing mode; inspect the declared schema and error policy.

Archived Databricks source — shown for reading, not Python execution.

```sql
USE CATALOG demo;
CREATE SCHEMA IF NOT EXISTS data_security;
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

## Learning notes — before source operation

**schema**: Compare declared or inferred field types with raw input. Parsing, schema evolution and rescued/corrupt records can produce nulls or additional columns rather than clean typed rows.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Parsing a numeric field from an incompatible string can fail or yield null depending on parsing mode; inspect the declared schema and error policy.

Archived Databricks source — shown for reading, not Python execution.

```sql
USE SCHEMA data_security;
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

#### Step 2: Create the table
A table is defined to represent sales data across multiple regions.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TABLE sales_dynamic (
  id INT,
  region STRING,
  email STRING,
  revenue INT
);
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

#### Step 3: Insert sample data
The dataset includes records from both UK and US regions.

This enables validation of row-level security by ensuring that different users see only relevant regional data.

## Learning notes — before source operation

**write**: This operation changes persistent state. Identify append, overwrite or incremental-copy semantics before rerunning; archived output does not establish the current destination contents.

Archived Databricks source — shown for reading, not Python execution.

```sql
INSERT INTO sales_dynamic VALUES
  (1, 'UK', 'john.doe@email.com', 10000),
  (2, 'US', 'mike.smith@email.com', 20000),
  (3, 'UK', 'sara.jones@email.com', 15000),
  (4, 'US', 'david.brown@email.com', 25000),
  (5, 'UK', 'emma.wilson@email.com', 18000);
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_inserted_rows
5	5

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM sales_dynamic;
```

In [ ]:
# Archived output only; no computation.

Archived table:
id	region	email	revenue
1	UK	john.doe@email.com	10000
2	US	mike.smith@email.com	20000
3	UK	sara.jones@email.com	15000
4	US	david.brown@email.com	25000
5	UK	emma.wilson@email.com	18000

#### Step 4: Create the dynamic view
The dynamic view implements both row-level security and column-level masking.

##### Row-Level Security

- UK group → returns only rows where region = 'UK'  
- US group → returns only rows where region = 'US'  
- Admin group → returns all rows  

##### Column-Level Masking

- Admin group → full email visible  
- Other users → email is masked  

##### Key Function: is_account_group_member()

This function evaluates whether the current user belongs to a specified group.

Examples:

- is_account_group_member('uk-sg') 
- is_account_group_member('us-sg') 
- is_account_group_member('admin-sg')  

The function returns TRUE or FALSE and is used to drive conditional logic inside the view.

## Learning notes — before source operation

**security**: Access depends on the executing identity, grants, filters/masks and storage configuration. A saved result is not evidence that another identity has the same access; credentials are not reproduced here.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT is_account_group_member('admin-sg'), is_account_group_member('uk-sg')
```

In [ ]:
# Archived output only; no computation.

Archived table:
is_account_group_member(admin-sg)	is_account_group_member(uk-sg)
False	True

## Learning notes — before source operation

**security**: Access depends on the executing identity, grants, filters/masks and storage configuration. A saved result is not evidence that another identity has the same access; credentials are not reproduced here.

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE VIEW vw_sales_dynamic AS
SELECT id,
       region,
       CASE 
         WHEN is_account_group_member('admin-sg') THEN email
         ELSE concat(substr(email, 1, 1), '***@', split(email, '@')[1])
       END AS email,  
       revenue
FROM sales_dynamic
WHERE is_account_group_member('admin-sg')
OR (is_account_group_member('uk-sg') AND region = 'UK')
OR (is_account_group_member('us-sg') AND region = 'US');
```

In [ ]:
# Archived output only; no computation.

Archived MIME payload:
{
  "application/vnd.databricks.empty-table+json": {
    "directive_name": "NoDirective"
  }
}

#### Step 5: Query the dynamic view

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM vw_sales_dynamic;
```

In [ ]:
# Archived output only; no computation.

Archived table:
id	region	email	revenue
1	UK	j***@email.com	10000
3	UK	s***@email.com	15000
5	UK	e***@email.com	18000